# TAO BYOM Image Classification

Transfer learning is the process of transfering learned features from one application to another. It is a commonly used training technique where you use a model trained on one task and re-train to use it on a different task. 

This notebook provides a complete life cycle of the model training, optimization and benchmarking using [NVIDIA TAO Toolkit](https://developer.nvidia.com/tao-toolkit). NVIDIA Train Adapt Optimize (TAO) Toolkit is a simple and easy-to-use Python based AI toolkit for taking purpose-built AI models and customizing them with users' own data.

**Bring Your Own Model (BYOM)** is a Python based package that converts any open source ONNX model to TAO compatible model. All you need is to export any model from a deep learning framework of your choice (e.g. PyTorch) to ONNX and run TAO BYOM converter. More details of installing and running TAO BYOM converter can be found [here](https://github.com/NVIDIA-AI-IOT/tao_byom_examples)

<br>

<img style="float: center; background-color: white; width: 1080" src="https://d29g4g2dyqv443.cloudfront.net/sites/default/files/akamai/TAO/tlt-tao-toolkit-bring-your-own-model-diagram.png" width="1080">
<br>

## Learning Objectives
In this notebook, you will learn how to leverage the simplicity and convenience of TAO to:

* Take a pretrained mobilenetv2 model and finetune on a TF flower dataset  to perform classification task.
* Prune the finetuned model.
* Inference check and Evaluation of finetuned model.
* Retrain the pruned model to recover lost accuracy.
* Export the pruned model as an onnx model.
* Inference check and Evaluation of onnx model.
* Convertion from onnx to tflite
* Post training quantization (PTQ).
* Inference check and Evaluation of tflite model.

At the end of this notebook, you will have generated a trained and optimized `classification` open-source model.
### Table of Contents
This notebook shows an example use case for classification using the Train Adapt Optimize (TAO) Toolkit.

0.  [Set up env variables and map drives](#head-0)
1.  [Installing the TAO Launcher](#head-1)
2.  [Prepare dataset and pretrained model](#head-2)<br>
    2.1 [Convert .onnx to .tltb with TAO-Byom](#head-2-1)<br>
    2.2 [Check the BYOM dataset format](#head-2-2)<br>
3.  [Provide training specification and generalize as per new model and dataset.](#head-3)<br>
4.  [Run TAO training](#head-4)<br>
5.  [Test Trained Model](#head-5)<br>
    5.1 [Evaluation Of Trained Model ](#head-5-1)<br>
6. [Inference Of Trained Model ](#head-6)<br>
    6.1 [Model Evaluation based on inference data](#head-6-1)<br>
7. [Pruning](#head-7)<br>   
8. [Retrain pruned model](#head-8)<br>
    8.1 [Updating spec.yaml with latest Model checkpoints](#head-8-1)<br>
9. [Test Retrained model](#head-9)<br>
    9.1 [Evaluation For Retrained Model](#head-9-1)<br>
10. [Inference Of Retrained Model](#head-10)<br>
    10.1 [Model Evaluation based on inference data](#head-10-1)<br>
11. [Exporting to ONNX model](#head-11)<br>
    11.1 [Running Inference and Evaluation For ONNX model](#head-11-1)<br>
12. [Exporting to TfLite model](#head-12)<br>
    12.1 [Converting ONNX Model to h5 format](#head-12-1)<br>
    12.2 [Converting h5 Model to tflite format(PTQ)](#head-12-2)<br>
    12.3 [Running Inference and Evaluation For TfLite Model](#head-12-3)<br>
13. [Converting TFlite to CC file ](#head-13)<br>

## 0. Set up env variables and map drives <a class="anchor" id="head-0"></a>
When using the purpose-built pretrained models from NGC, please make sure to set the `$KEY` environment variable to the key as mentioned in the model overview. Failing to do so, can lead to errors when trying to load them as pretrained models.

The following notebook requires the user to set an env variable called the `$LOCAL_PROJECT_DIR` as the path to the users workspace. Please note that the dataset to run this notebook is expected to reside in the `$LOCAL_PROJECT_DIR/data`, while the TAO experiment generated collaterals will be output to `$LOCAL_PROJECT_DIR/classification_tf2`. More information on how to set up the dataset and the supported steps in the TAO workflow are provided in the subsequent cells.

*Note: Please make sure to remove any stray artifacts/files from the `$USER_EXPERIMENT_DIR` or `$DATA_DOWNLOAD_DIR` paths as mentioned below, that may have been generated from previous experiments. Having checkpoint files etc may interfere with creating a training graph for a new experiment.*

*Note: This notebook currently is by default set up to run training using 1 GPU. To use more GPU's please update the env variable `$NUM_GPUS` accordingly*

In [ ]:
%use venv_tf2_9_1
import os
# Setting up env variables for cleaner command line commands.

%env KEY=nvidia_tlt
%env NUM_GPUS=1
%env USER_EXPERIMENT_DIR=/workspace/tao-experiments/byom/
%env DATA_DOWNLOAD_DIR=/workspace/tao-experiments/byom/data

# Set this path if you don't run the notebook from the samples directory.
# %env NOTEBOOK_ROOT=~/tao-samples/classification_tf2.

# Please define this local project directory that needs to be mapped to the TAO docker session.
# The dataset expected to be present in $LOCAL_PROJECT_DIR/data, while the results for the steps
# in this notebook will be stored at $LOCAL_PROJECT_DIR/classification_tf2/byom_person/results_mobilenetv2/
# !PLEASE MAKE SURE TO UPDATE THIS PATH!.

os.environ["LOCAL_PROJECT_DIR"] = os.path.dirname(os.getcwd())
os.environ["LOCAL_EXPERIMENT_DIR"] = os.path.join(os.environ["LOCAL_PROJECT_DIR"],os.getcwd().split("/")[-1])

In [ ]:
# creating required directory for different stages
!mkdir -p $LOCAL_EXPERIMENT_DIR/export
!mkdir -p $LOCAL_EXPERIMENT_DIR/results
!mkdir -p $LOCAL_EXPERIMENT_DIR/data
!mkdir -p $LOCAL_EXPERIMENT_DIR/results/retrain
!mkdir -p $LOCAL_EXPERIMENT_DIR/pretrained_model
!mkdir -p $LOCAL_EXPERIMENT_DIR/results/retrain/export

In [ ]:
# creating environment variables for mapping directories

os.environ["LOCAL_DATA_DIR"]    = os.path.join(os.environ.get("NOTEBOOK_ROOT", os.getcwd()),"data")

# The sample spec files are present in the same path as the downloaded samples.
os.environ["LOCAL_SPECS_DIR"] = os.path.join(os.environ.get("NOTEBOOK_ROOT", os.getcwd()),"specs")
os.environ["LOCAL_RESULTS_DIR"] = os.path.join(os.environ.get("NOTEBOOK_ROOT", os.getcwd()),"results")

%env SPECS_DIR=/workspace/tao-experiments/byom/specs
%env RESULTS_DIR=/workspace/tao-experiments/byom/results
%env DATA_DIR=/workspace/tao-experiments/byom/data

# to store the variable for venv_gui
LOCAL_EXPERIMENT_DIR = os.environ.get("LOCAL_EXPERIMENT_DIR")
%store LOCAL_EXPERIMENT_DIR 

# 
KEY = os.environ.get("KEY")
%store KEY

# Showing list of specification files.
!ls -rlt $LOCAL_SPECS_DIR

In [ ]:
%use venv_tf2_9_1

# importing all required packages
import os
import gc
import json
import yaml
import time
import shutil
import random
import pathlib
import subprocess 
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns
from tqdm import tqdm
import tensorflow as tf
import onnxruntime as ort
import matplotlib.pyplot as plt
from difflib import get_close_matches
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

## Import all the required packages and functions.

The cell below maps the project directory on your local host to a workspace directory in the TAO docker instance, so that the data and the results are mapped from outside to inside of the docker instance.

In [ ]:

# Mapping up the local directories to the TAO docker.

mounts_file = os.path.expanduser("~/.tao_mounts.json")


# Define the dictionary with the mapped drives
drive_map = {
    "Mounts": [
        # Mapping the project directory
        {
            "source": os.environ["LOCAL_PROJECT_DIR"],
            "destination": "/workspace/tao-experiments"
        },
        # Mapping the specs directory.
        {
            "source": os.environ["LOCAL_SPECS_DIR"],
            "destination": os.environ["SPECS_DIR"]
        },
        # Mapping the result directory.
        {
            "source": os.environ["LOCAL_RESULTS_DIR"],
            "destination": os.environ["RESULTS_DIR"]
        },
        # Mapping the data directory.
        {
            "source": os.environ["LOCAL_DATA_DIR"],
            "destination": os.environ["DATA_DIR"]
        },
    ],
    "DockerOptions":{
        "user": "{}:{}".format(os.getuid(), os.getgid())
    }
}

# Writing the mounts file.
with open(mounts_file, "w") as mfile:
    json.dump(drive_map, mfile, indent=4)

In [ ]:

!cat ~/.tao_mounts.json

## 1. Installing the TAO launcher <a class="anchor" id="head-1"></a>
The TAO launcher is a python package distributed as a python wheel listed in PyPI. You may install the launcher by executing the following cell.

Please note that TAO Toolkit recommends users to run the TAO launcher in a virtual env with python 3.6.9. However, **to perform onnx quantization and evaluation using the provided scripts, we recommend to use Python version 3.8.16**. This notebooks is tested for Python version 3.8.16. You may follow the instruction in this [page](https://virtualenvwrapper.readthedocs.io/en/latest/install.html) to set up a python virtual env using the `virtualenv` and `virtualenvwrapper` packages. Once you have setup virtualenvwrapper, please set the version of python to be used in the virtual env by using the `VIRTUALENVWRAPPER_PYTHON` variable. You may do so by running

```sh
export VIRTUALENVWRAPPER_PYTHON=/path/to/bin/python3.x
```
where x >= 6 and <= 8.16

We recommend performing this step first and then launching the notebook from the virtual environment. In addition to installing TAO python package, please make sure of the following software requirements:
* python >=3.6.9 <= 3.8.x
* docker-ce > 19.03.5
* docker-API 1.40
* nvidia-container-toolkit > 1.3.0-1
* nvidia-container-runtime > 3.4.0-1
* nvidia-docker2 > 2.5.0-1
* nvidia-driver > 455+

Once you have installed the pre-requisites, please log in to the docker registry nvcr.io by following the command below

```sh
docker login nvcr.io
```

You will be triggered to enter a username and password. The username is `$oauthtoken` and the password is the API key generated from `ngc.nvidia.com`. Please follow the instructions in the [NGC setup guide](https://docs.nvidia.com/ngc/ngc-overview/index.html#generating-api-key) to generate your own API key.


In [ ]:
!echo "skip: pip3 install nvidia-tao   (already installed)"
#pip3 install nvidia-tao

In [ ]:
# printing tao info
!tao info

## 2. Prepare datasets and pre-trained model <a class="anchor" id="head-2"></a>
converting onnx model to tltb format and download the data set and check the dataset is in correct format.

### 2.1 Convert .onnx to .tltb with TAO-Byom<a class="anchor" id="head-2-2"></a>
This step is to create the .tltb file from the .onnx file. .tltb is an internal format of TAO and this step is needed to start training using TAO. This step will create a byom_model directory with the file and complementary information for this in it.

* Used pretrained model mobilenet v2 from the <a href="https://github.com/onnx/models/tree/main/validated/vision/classification/mobilenet">Model link</a>.
* Used MobileNet v2-1.0 with model size 13.3mb and top1 accuracy as 70.94. onnx version 1.5.0 and opset version as 10.
* The pretrained model was created using MXNet framework and trined on Imagenet with 1000 labels.

If we wish to fine tune the pretrained model (as is the case) on a different dataset through TAO Toolkit, we must remove the classification head for ImageNet. Hence, the final converted TAO model should only contain layers up to the penultimate layer, which is a layer before the average pooling. In this case, the node name is `463`. Adding `-p 463` argument, removes the head of the model while creatig the `.tltb` file.

`Note` : to change the model replace below code with the new model link or import the downloaded model to pretrained_model folder.

* Use netron app to find out the head value for `-p`.
* User can change the downloaded onnx model name by using `onnx_modelname` and new model download link through `model_download_link` .
* User can change the converted tltb model name by using `tltb_modelname`.

In [ ]:
# creating an env variable for onnx model name and download link.
import os 
%store -r LOCAL_EXPERIMENT_DIR
%env onnx_modelname = mobilenet_v2.onnx
%env tltb_modelname = mobilenet_conv
tltb_modelname = "mobilenet_conv.tltb"
%store tltb_modelname
%env model_download_link = https://github.com/onnx/models/raw/refs/heads/main/validated/vision/classification/mobilenet/model/mobilenetv2-10.onnx

In [ ]:
# creating an env variable for onnx model path 
os.environ["pretrained_Modelpath"] = os.path.join(LOCAL_EXPERIMENT_DIR,"pretrained_model")
os.environ["pretrained_onnx_Modelpath"] = os.path.join(LOCAL_EXPERIMENT_DIR,"pretrained_model",os.environ["onnx_modelname"])

In [ ]:

# Downloading the model from the git hub
!wget -O $pretrained_onnx_Modelpath  $model_download_link

In [ ]:

# creating the .tltb file from the pretrained onnx file
%env TF_ENABLE_ONEDNN_OPTS=1
!echo "tao_byom -m $pretrained_onnx_Modelpath -k nvidia_tlt -r $pretrained_Modelpath -n $tltb_modelname -p 463"
!tao_byom -m $pretrained_onnx_Modelpath -k nvidia_tlt -r $pretrained_Modelpath -n $tltb_modelname -p 463

In [ ]:
# printing the downloaded onnx model and converted tltb model 
print("Check that BYOM model was converted into the dir.")
!ls -l $LOCAL_EXPERIMENT_DIR/pretrained_model

### 2.2 Check the BYOM dataset format <a class="anchor" id="head-2-2"></a>
Here, tf_flowers dataset from tensorflow dataset with 5400 images in total with minimum of 1000 images for each labels. done split based on train, test, and valid with 70% for train and 15% for test and 15% for train each split contains 5 labels of flowers (daisy,dandelion,roses,sunflowers,tulips).
<a href="https://www.tensorflow.org/datasets/catalog/tf_flowers">Dataset link</a></br></br>
We will be using the classification dataset in the project directory data. with train,valid and test subsets.
Verifying if the portions are created and all the sub-folders have all the classes.
* Training dataset   - used 70% of data with 3345 image in total with minumum of 570 images for each class.
* Validation dataset - used 15% of data with 1025 image in total with minumum of 170 images for each class.
* Test dataset       - used 15% of data with 1025 image in total with minumum of 170 images for each class.

`Note` : User can replace custom dataset in `./data` folder like the folder structure given below.
 - data
     - train
       - class-A
       - class-B
     - test
       - class-A
       - class-B
     - valid
       - class-A
       - class-B
     - infer

In [ ]:
# downloading the dataset form tensorflow datasets.
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
custom_dir = os.environ["LOCAL_EXPERIMENT_DIR"]
archive = tf.keras.utils.get_file(origin=dataset_url, extract=True, cache_dir=custom_dir)
data_dir = pathlib.Path(archive).with_suffix('')
os.remove(archive)

In [ ]:
# splitting the dataset into train test and valid.
import os
def split_data(source,target,train_ratio=0.7,val_ratio=0.15,test_ratio=0.15):
    classess=[folder for folder in os.listdir(source) if os.path.isdir(os.path.join(source,folder))]
    for cls in classess:
        class_path=os.path.join(source,cls)
        images=os.listdir(class_path)
        random.shuffle(images)
        
        train_end=int(len(images)*train_ratio)
        val_end=train_end +int(len(images)*val_ratio)
        splits={
            'train':images[:train_end],
            'valid':images[train_end:val_end],
            'test':images[val_end:]
        }
        for split in splits:
            split_dir=os.path.join(target,split,cls)
            os.makedirs(split_dir,exist_ok=True)
            for img in splits[split]:
                src=os.path.join(class_path,img)
                dst=os.path.join(split_dir,img)
                shutil.copy2(src,dst)

# modify this ratio as per the user requirements.
train_ratio=0.7
val_ratio=0.15
test_ratio=0.15

source=os.path.join(os.environ["LOCAL_EXPERIMENT_DIR"],"datasets/flower_photos")
target=os.path.join(os.environ["LOCAL_EXPERIMENT_DIR"],"data")
split_data(source,target,train_ratio,val_ratio,test_ratio)
print(f"dataset split was sucessfull with train-{train_ratio*100}%,test-{test_ratio*100}% and valid-{val_ratio*100}%")

In [ ]:
# create inferance folder with images.
def create_infer(source_dir,validation_dir):
    os.makedirs(validation_dir, exist_ok=True)
    for label in os.listdir(source_dir):
        label_dir = os.path.join(source_dir, label)
        if os.path.isdir(label_dir):
            for idx, fname in enumerate(os.listdir(label_dir)):
                if idx < 100:
                    src_file = os.path.join(label_dir, fname)
                    if os.path.isfile(src_file) and fname.lower().endswith(('.jpg', '.png', '.jpeg')):
                        # Create new file name with label and index
                        new_file_name = f'{label}_{idx}.jpg'
                        dst_file = os.path.join(validation_dir, new_file_name)
                        shutil.copy(src_file, dst_file)

# Create validation directory if it doesn't exist

source_dir = os.path.join(os.environ["LOCAL_EXPERIMENT_DIR"],"datasets/flower_photos")
validation_dir = os.path.join(os.environ["LOCAL_EXPERIMENT_DIR"],"data/infer")
#function call
create_infer(source_dir,validation_dir)                   
!rm -r datasets
print("inference dataset was created..!!")

In [ ]:

# printing the input data format
!tree -n $LOCAL_DATA_DIR -d

## 3. Provide training specification and generalize as per new model and dataset.<a class="anchor" id="head-3"></a>

* Below are the training parameters to be changed as per new model.
* num_classes : number of classes as per the training dataset. 
* preprocess_mode(tf,caffe,torch) : torch is currently used as a preprocess mode. user can change as per the model selected.
* input_height, input_width : update as per the model input.(make use of netron app for getting input shape).
* backbone : update as per the model selected.
* optimizer : optimizer can be selected based on fine tuning parameters.
* num_epochs : train the model for more epochs to achieve better accuracy.
* batch_size : Number of images per training. Depends on GPU memory.

`Note` : Dynamic update of below training spec parameters is reflected on the training spec file. For advanced parameters user can change `./specs/spec.yaml` as per the requirements. 

In [ ]:

%store -r tltb_modelname

# updating spec file as per the new model requirement.
spec_file_path = os.path.join(os.environ["LOCAL_SPECS_DIR"],"spec.yaml")
with open(spec_file_path) as fp:
    try:
        data = yaml.safe_load(fp)
    except yaml.YAMLError as exc:
        print(exc)

tltb_file_path = os.path.join(os.environ["USER_EXPERIMENT_DIR"],"pretrained_model",f"{tltb_modelname}")

# updating the training specs 
data["results_dir"] = os.environ["RESULTS_DIR"]

data["dataset"]["num_classes"]         = 5
data["dataset"]["preprocess_mode"]     = "torch"
data["dataset"]["train_dataset_path"]  = os.path.join(os.environ["DATA_DIR"],"train")
data["dataset"]["val_dataset_path"]    = os.path.join(os.environ["DATA_DIR"],"valid")

data["model"]["input_width"]     = 224
data["model"]["input_height"]    = 224
data["model"]["input_channels"]  = 3
data["model"]["byom_model"]      = tltb_file_path
data["model"]["backbone"]        = "mobilenet_v2"


data["train"]["checkpoint"]  = ""
data["train"]["num_epochs"]  = 1
data["train"]["optim_config"]["optimizer"] = "adam"

data["evaluate"]["batch_size"]    = 16
data["evaluate"]["dataset_path"]  = os.path.join(os.environ["DATA_DIR"],"test")

data["inference"]["classmap"]   = os.path.join(os.environ["RESULTS_DIR"],"train","classmap.json")
data["inference"]["image_dir"]  = os.path.join(os.environ["DATA_DIR"],"infer")

with open(spec_file_path, 'w') as fp:
    data = yaml.dump(data,fp,sort_keys=False)

print("successfully updated the spec.yaml as per the new model training requirements.....!!")

In [ ]:

#printing the updated training configuration
!cat $LOCAL_SPECS_DIR/spec.yaml

## 4. Run TAO training <a class="anchor" id="head-4"></a>
* Based on the sample spec file and the output directory model are saved in `./results/train` folder.


`Note` : The training can be intrupted and then relaunched at any point. To resume from a checkpoint, just relaunch training with the same spec file.

In [ ]:
# logging in the docker session
!echo "skip not required done by environment files. !docker login nvcr.io"
#!docker login nvcr.io

In [ ]:
# running training commands
!tao model classification_tf2 train -e $SPECS_DIR/spec.yaml 

In [ ]:
# printing the trained models
print("Trained models.")
!ls -l $LOCAL_EXPERIMENT_DIR/results/train

## 5. Test Trained Model <a class="anchor" id="head-5"></a>

In this step, we assume that the training is complete and the resulting weights for the model are available. this script below will find the last checkpoint corresponding to the latest finished epoch and run the evaluation on that, by passing it to the configuration parameter `evaluate.checkpoint` and `inference.checkpoint`.

* Updating the latest model checkpoint path for evaluate and inference. dynamic update of latest model checkpoint in spec.yaml

In [ ]:

# updating last checkpoint to spec.yaml
file_path = os.path.join(os.environ["LOCAL_SPECS_DIR"],"spec.yaml")
with open(file_path) as fp:
    try:
        data = yaml.safe_load(fp)
    except yaml.YAMLError as exc:
        print(exc)
        
last_checkpoint = ''

for file_name in os.listdir(os.path.join(os.environ["LOCAL_RESULTS_DIR"], 'train')):
    if file_name.startswith(str(data['model']['backbone'])):
        last_checkpoint = last_checkpoint if last_checkpoint > file_name else file_name
        
print(f'Last checkpoint: {last_checkpoint}')
%env LAST_CHECKPOINT={last_checkpoint}


last_checkpoint_path = os.path.join(os.environ["RESULTS_DIR"],"train",os.environ["LAST_CHECKPOINT"])

# updating latest checkpoints 
data["evaluate"]["checkpoint"]  = last_checkpoint_path
data["inference"]["checkpoint"] = last_checkpoint_path

with open(file_path, 'w') as fp:
    data = yaml.dump(data,fp,sort_keys=False)

print("successfully updated the spec.yaml with latest checkpoint for evaluate and inference....", os.environ["LAST_CHECKPOINT"])

###    5.1 Evaluation Of Trained Model <a class="anchor" id="head-5-1"></a>
Evaluating the classification model. Model can be evaluated based on the Confussion matrix obtained.

In [ ]:
#executing evaluation code
!tao model classification_tf2 evaluate -e $SPECS_DIR/spec.yaml 

## 6. Inference Of Trained Model <a class="anchor" id="head-6"></a>
Inference of classification model is exicuted and the results are saved in `results.csv` which can be located at `results/inference`. used data from infer folder present in data folder.

`Note` : user can add images for inference at `data/infer` folder

In [ ]:
# executing inference code
!tao model classification_tf2 inference -e  $SPECS_DIR/spec.yaml

### 6.1 Model Evaluation based on inference data <a class="anchor" id="head-6-1"></a>
printing the confusion matrix for evaluation based on the `results.csv` obtained in previous step.

In [ ]:
inferance_results_path = os.path.join(os.environ["LOCAL_RESULTS_DIR"],"inference/result.csv")

#parsing the path value to get the actual input
data = pd.read_csv(inferance_results_path,names=["path","pred","pred_score"])
data = data[:-1] 
data["act"] = data["path"].str.split("/").str[-1]
data["act"] = data["act"].str.split(".").str[0]
data["act"] = data["act"].str.replace(r'[^a-zA-Z]','',regex=True)
data["act"] = data["act"].str.lower()
data = data.dropna(subset=["act"])

actual_labels  = data["act"].unique()

#mapping the prediction value to same labels as actual
mapped_pred = []
for pred in data["pred"]:
    match = get_close_matches(pred,actual_labels,n=1, cutoff=0.6)
    mapped_pred.append(match[0] if match else pred)
data["pred_mapped"] = mapped_pred

# printing the classification report for the inference data
print("\n-------------------- INFERENCE RESULTS --------------------\n")
print(classification_report(data["act"],data["pred_mapped"]))

## 7. Pruning <a class="anchor" id="head-7"></a>

Usually, you just need to adjust `threshold` for accuracy and model size trade off. Higher `threshold` gives you smaller model (and thus higher inference speed) but lesser accuracy. The threshold to use is depend on the dataset. `0.5` is just a starting point.

`min_num_filters` is the minimum number of filters to keep per layer after the pruning step. Smaller the value smaller the resulting model. However, this might lower the accuracy. The users can adjust this value depending on their needs.

`Note:` The model was pruned on 0.85 treshold and used minimum number of filter as 16

In [ ]:

# updating the yaml file with new pruning specs.
file_path = os.path.join(os.environ["LOCAL_SPECS_DIR"],"spec.yaml")
with open(file_path) as fp:
    try:
        data = yaml.safe_load(fp)
    except yaml.YAMLError as exc:
        print(exc)
        
# updating the pruning specs 
data["prune"]["threshold"]       = 0.85
data["prune"]["checkpoint"]      = last_checkpoint_path
data["prune"]["min_num_filters"] = 16

with open(file_path, 'w') as fp:
    data = yaml.dump(data,fp,sort_keys=False)

print("successfully updated the spec.yaml with new pruning specifications.....!!!")

In [ ]:
# running tao code for pruning
!tao model classification_tf2 prune -e  $SPECS_DIR/spec.yaml 

In [ ]:
# printing the pruned model 
print('Pruned model:')
print('------------')
!ls -rlt $LOCAL_RESULTS_DIR/prune

## 8. Retrain pruned model <a class="anchor" id="head-8"></a>
`NOTE` : Model needs to be re-trained to bring back accuracy after pruning

The file name of the pruned model is created based on the `prune.threshold` value automatically. Path was automatically updated in `specs.yaml` based on the file name. Make sure to update the path as per the pruned model. 

### 8.1 Updating spec.yaml with latest Model checkpoints <a class="anchor" id="head-8-1"></a>
* Updating the pruned model path for retraining by updating under the train-checkpoint.

In [ ]:
# updating spec file
pruned_model = ''

for file_name in os.listdir(os.path.join(os.environ["LOCAL_RESULTS_DIR"], 'prune')):
    if file_name.startswith('model_th') and file_name.endswith('.tlt'):
        pruned_model = file_name
        
print(f'pruned_model: {pruned_model}')
%env PRUNED_MODEL = {pruned_model}

file_path         = os.path.join(os.environ["LOCAL_SPECS_DIR"],"spec.yaml")
pruned_model_path = os.path.join(os.environ["RESULTS_DIR"],"prune",os.environ["PRUNED_MODEL"])

with open(file_path) as fp:
    try:
        data = yaml.safe_load(fp)
    except yaml.YAMLError as exc:
        print(exc)

# updating pruned model as new pretrained model and switching new training path
data["train"]["checkpoint"]  = pruned_model_path
data["results_dir"] = os.path.join(os.environ["RESULTS_DIR"],"retrain")

with open(file_path, 'w') as fp:
    data = yaml.dump(data,fp,sort_keys=False)

print("successfully updated the spec.yaml with latest checkpoint for retraining ....", os.environ["PRUNED_MODEL"])

In [ ]:
# remove the status file 
# !rm -r $LOCAL_RESULTS_DIR/train/status.json
!cat $LOCAL_SPECS_DIR/spec.yaml

In [ ]:
# running retraining commands
!tao model classification_tf2 train -e $SPECS_DIR/spec.yaml

In [ ]:
# printing the re-trained models
print("Retrained models.")
!ls -l $LOCAL_EXPERIMENT_DIR/results/retrain/train

## 9. Test Retrained model! <a class="anchor" id="head-9"></a>

In this step, we assume that the training is complete and the model from the final epoch (`mobilenet_v2_xxx.tlt`) is available.
* Updating the latest model checkpoint path for evaluate and inference. dynamic update of latest model checkpoint in `spec.yaml`, changing the `training` specs with `retrain` specs.

In [ ]:
# updating last checkpoint to spec.yaml
file_path = os.path.join(os.environ["LOCAL_SPECS_DIR"],"spec.yaml")
last_checkpoint = ''

#loading spec.yaml
with open(file_path) as fp:
    try:
        data = yaml.safe_load(fp)
    except yaml.YAMLError as exc:
        print(exc)

for file_name in os.listdir(os.path.join(os.environ["LOCAL_RESULTS_DIR"], 'retrain/train')):
    if file_name.startswith(str(data['model']['backbone'])):
        last_checkpoint = last_checkpoint if last_checkpoint > file_name else file_name
        
print(f'Last checkpoint: {last_checkpoint}')
%env LAST_CHECKPOINT={last_checkpoint}

last_checkpoint_path = os.path.join(os.environ["RESULTS_DIR"],"retrain/train",os.environ["LAST_CHECKPOINT"])

# updating latest checkpoints 
data["evaluate"]["checkpoint"]  = last_checkpoint_path
data["inference"]["checkpoint"] = last_checkpoint_path

with open(file_path, 'w') as fp:
    data = yaml.dump(data,fp,sort_keys=False)

print("successfully updated the spec.yaml with latest checkpoint for evaluate and inference....", os.environ["LAST_CHECKPOINT"])

###    9.1 Evaluation For Retrained Model <a class="anchor" id="head-9-1"></a>
Evaluating the classification model. Model can be evaluated based on the Confussion matrix obtained.

In [ ]:
#run specification
!echo spec file: "$SPECS_DIR/spec.yaml"
!tao model classification_tf2 evaluate -e $SPECS_DIR/spec.yaml 

## 10. Inference of Retrained Model <a class="anchor" id="head-10"></a>

Inference of classification model is exicuted and the results are saved in `results.csv` which can be located at `results/retrain/inference`. Used data from infer folder present in data folder.

`Note` : user can add images for inference at `./data/infer` folder

In [ ]:
# run inference
!tao model classification_tf2 inference -e  $SPECS_DIR/spec.yaml                                 

### 10.1 Model Evaluation based on inference data <a class="anchor" id="head-10-1"></a>
printing the confusion matrix for Evaluation based on the `results.csv` obtained in previous step.

In [ ]:
%use venv_tf2_9_1

inferance_results_path = os.path.join(os.environ["LOCAL_RESULTS_DIR"],"retrain/inference/result.csv")

#parsing the path value to get the actual input
data = pd.read_csv(inferance_results_path,names=["path","pred","pred_score"])
data = data[:-1] 
data["act"] = data["path"].str.split("/").str[-1]
data["act"] = data["act"].str.split(".").str[0]
data["act"] = data["act"].str.replace(r'[^a-zA-Z]','',regex=True)
data["act"] = data["act"].str.lower()
data = data.dropna(subset=["act"])

actual_labels  = data["act"].unique()

#mapping the prediction value to same labels as actual
mapped_pred = []
for pred in data["pred"]:
    match = get_close_matches(pred,actual_labels,n=1, cutoff=0.8)
    mapped_pred.append(match[0] if match else pred)
data["pred_mapped"] = mapped_pred

# printing the classification report for the inference data
print("\n-------------------- INFERENCE RESULTS --------------------\n")
print(classification_report(data["act"],data["pred_mapped"]))

## 11. Exporting the Retrained model! <a class="anchor" id="head-11"></a>
The following section exports the pruned and retrained model as an onnx file.
* Updating the latest model checkpoint path for exporting the model as ONNX. dynamic update of latest model checkpoint in `spec.yaml`
* user can provide the name for exporting onnx model using `export_onnx_modelname`.

In [ ]:
# setting name for the exporting onnx model
%env export_onnx_modelname = export_model.onnx

file_path = os.path.join(os.environ["LOCAL_SPECS_DIR"],"spec.yaml")
last_checkpoint = ''

with open(file_path) as fp:
    try:
        data = yaml.safe_load(fp)
    except yaml.YAMLError as exc:
        print(exc)

for file_name in os.listdir(os.path.join(os.environ["LOCAL_RESULTS_DIR"], 'retrain/train')):
    if file_name.startswith(str(data['model']['backbone'])):
        last_checkpoint = last_checkpoint if last_checkpoint > file_name else file_name

        
print(f'Last checkpoint: {last_checkpoint}')
%env LAST_CHECKPOINT={last_checkpoint}


last_checkpoint_path = os.path.join(os.environ["RESULTS_DIR"],"retrain/train",os.environ["LAST_CHECKPOINT"])

# updating the export configurations.
data["export"]["onnx_file"]   = os.path.join(os.environ["RESULTS_DIR"],'retrain/export',os.environ["export_onnx_modelname"])
data["export"]["results_dir"] = os.path.join(os.environ["RESULTS_DIR"],'retrain/export')
data["export"]["checkpoint"]  = last_checkpoint_path

with open(file_path, 'w') as fp:
    data = yaml.dump(data,fp,sort_keys=False)

print("successfully updated the spec.yaml with latest checkpoint for evaluate and inference....", os.environ["LAST_CHECKPOINT"])

In [ ]:

# exporting the model to onnx format
!tao model classification_tf2 export -e  $SPECS_DIR/spec.yaml                       

In [ ]:
# printing the exported onnx model
print('Exported ONNX model:')
print('------------')
!ls -rlt $LOCAL_RESULTS_DIR/retrain/export

###    11.1 Running Inference and Evaluation For ONNX Model <a class="anchor" id="head-11-1"></a>
running the inference of onnx model and evaluating the onnx model on test data. confusion matrix is plotted as evaluation result.

In [ ]:
# running inference code for plotting heatmat and classification report
data_root = os.path.join(os.environ["LOCAL_DATA_DIR"],"test") # Folder structure: test_dataset/class_name/image.jpg
class_names = sorted(os.listdir(data_root))  # Auto-detect class names
class_to_index = {name: idx for idx, name in enumerate(class_names)}
 
# Preprocessing for torch
def preprocess_image_torch(img_path,height,width):
    img = Image.open(img_path).convert('RGB').resize((height, width))
    img = np.array(img).astype('float32') / 255.0
    img = (img - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    img = np.transpose(img, (2, 0, 1))  # CHW
    img = np.expand_dims(img, axis=0)
    return img
 
# Load all image paths and labels

image_paths = []
labels = []
for class_name in class_names:
    class_dir = os.path.join(data_root, class_name)
    for img_file in os.listdir(class_dir):
        if img_file.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(os.path.join(class_dir, img_file))
            labels.append(class_to_index[class_name])


In [ ]:
# evaluating ONNX model based on test data
file_path = os.path.join(os.environ["LOCAL_SPECS_DIR"],"spec.yaml")

with open(file_path) as fp:
    try:
        data = yaml.safe_load(fp)
    except yaml.YAMLError as exc:
        print(exc)

# onnx local model path
onnx_model_path = os.path.join(os.environ["LOCAL_RESULTS_DIR"],"retrain/export",os.environ["export_onnx_modelname"])

ort_session = ort.InferenceSession(onnx_model_path)
onnx_preds = []

if data["dataset"]["preprocess_mode"] == 'torch' :
    for path in image_paths:
        input_tensor = preprocess_image_torch(path,data["model"]["input_height"],data["model"]["input_width"])
        input_tensor = input_tensor.astype(np.float32)
        ort_inputs = {ort_session.get_inputs()[0].name: input_tensor}
        ort_outs = ort_session.run(None, ort_inputs)
        pred_label = np.argmax(ort_outs[0])
        onnx_preds.append(pred_label)
 
print("\n ONNX Inference:")
print("Top-1 Accuracy:", accuracy_score(labels, onnx_preds))
print("Classification Report:\n", classification_report(labels, onnx_preds, target_names=class_names))
cm = confusion_matrix(labels, onnx_preds)
sns.heatmap(cm, annot=True, cmap='Blues', fmt='d', xticklabels=class_names, yticklabels=class_names)
plt.title("ONNX Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

##    12. Convert ONNX Model To Tflite <a class="anchor" id="head-12"></a>
current section flow as shown below 
* onnx model -> h5 -> tflite(ptq) 

###    12.1 Converting ONNX Model to h5 format <a class="anchor" id="head-12-1"></a>
`note` : used code from pinto github page for conversion of ONNX to h5 format. <a href="https://github.com/PINTO0309/onnx2tf">GitHub link</a>


In [ ]:
# 
!docker run --rm -it -v $LOCAL_RESULTS_DIR/retrain/export:/workdir \
                     -w /workdir docker.io/pinto0309/onnx2tf:1.26.8 onnx2tf \
                     -i export_model.onnx -oh5

In [ ]:
# printing the converted model in h5 format.
print('Converted pb model:')
print('------------')
!ls -rlt $LOCAL_RESULTS_DIR/retrain/export/saved_model

###    12.2 Converting h5 Model to tflite format (PTQ) <a class="anchor" id="head-12-2"></a>
Validataion data used for calibration. `max_images_per_class = 150` : this fetches 150 images per class from the validation dataset. The user can modify as per different model choosen for calibration.
* user can provide the name for exporting tflite model using `export_tflite_modelname`.

In [ ]:

def convert_h5_to_tflite(saved_model_path, tflite_model_path, dataset_path, max_images_per_class):

    file_path = os.path.join(os.environ["LOCAL_SPECS_DIR"],"spec.yaml")

    with open(file_path) as fp:
        try:
            data = yaml.safe_load(fp)
        except yaml.YAMLError as exc:
            print(exc)

    model = tf.saved_model.load(saved_model_path)
    concrete_func = model.signatures["serving_default"]
    concrete_func.inputs[0].set_shape([1,data["model"]["input_height"],data["model"]["input_width"],data["model"]["input_channels"]])
    # Create the converter from SavedModel
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    image_size = (data["model"]["input_height"],data["model"]["input_width"])
 
    def preprocess_for_quantization_torch(image_path):
        image = Image.open(image_path).resize(image_size).convert("RGB")
        img_array = np.array(image).astype(np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img_array = (img_array - mean) / std
        return np.expand_dims(img_array, axis=0)
 
    def representative_dataset_gen():
        image_paths = []
        for class_name in sorted(os.listdir(dataset_path)):
            class_dir = os.path.join(dataset_path, class_name)
            if not os.path.isdir(class_dir):
                continue
            images = sorted(os.listdir(class_dir))[:max_images_per_class]
            for image_name in images:
                image_paths.append(os.path.join(class_dir, image_name))
 
        print(f"Using {len(image_paths)} images for representative dataset")

        if data["dataset"]["preprocess_mode"] == 'torch' :
            for path in image_paths:
                try:
                    input_data = preprocess_for_quantization_torch(path).astype(np.float32)
                    yield [input_data]
                except Exception as e:
                    print(f"Skipping {path}: {e}")
 
    # Enable optimization and INT8 quantization settings

    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
 
    # Perform the conversion and write to output file
    tflite_model = converter.convert()
    with open(tflite_model_path, 'wb') as f:
        f.write(tflite_model)
    print(f" Conversion completed: {tflite_model_path}")
    

In [ ]:
# add model name to env variable

%env export_tflite_modelname = export_model.tflite

# running convertion function.i
import os 
convert_h5_to_tflite(os.path.join(os.environ["LOCAL_EXPERIMENT_DIR"],"results/retrain/export/saved_model"), \
                     os.path.join(os.environ["LOCAL_RESULTS_DIR"],"retrain/export",os.environ["export_tflite_modelname"]),\
                     os.path.join(os.environ["LOCAL_DATA_DIR"],"valid"),150)

In [ ]:
# printing the converted model in tflite format.
print('Converted tflite model:')
print('------------')
!ls -rlt $LOCAL_RESULTS_DIR/retrain/export/

`note` : for MCU RA8D1 Board, the obtained tflite model should be less than 1mb.

###    12.3 Running Inference and Evaluation For TfLite Model <a class="anchor" id="head-12-3"></a>
running the inference of tflite model and evaluating the tflite model on test data. confusion matrix is plotted as evaluation result.

In [ ]:
# setting directory and model path
test_dir = os.path.join( os.environ["LOCAL_EXPERIMENT_DIR"], "data/test")
model_path = os.path.join( os.environ["LOCAL_RESULTS_DIR"],"retrain/export", os.environ["export_tflite_modelname"])

def reset_backend():
    tf.keras.backend.clear_session()
    tf.keras.backend.set_image_data_format("channels_last")
    tf.keras.backend.set_floatx("float32")
    tf.keras.backend.set_epsilon(1e-7)
    import gc; gc.collect()

def preprocess_image_torch(img_path,h,w):
    img = Image.open(img_path).convert('RGB').resize((h, w))
    img = np.array(img).astype('float32') / 255.0
    img = (img - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    img = np.expand_dims(img, axis=0)
    return img
    
def inference_tflite():
    # print(f"\n--- Starting Inference: {os.path.basename(model_path)} ---")
    
    # Load TFLite model
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    input_shape = input_details[0]['shape']
    h, w = input_shape[1], input_shape[2]
    
    class_names = sorted(os.listdir(test_dir)) 
    class_to_index = {name: idx for idx, name in enumerate(class_names)}
    labels = []
    prediction = []
    
    for class_name in os.listdir(test_dir):
        class_path = os.path.join(test_dir, class_name)
        if not os.path.isdir(class_path):
            continue
        
        for image_name in tqdm(os.listdir(class_path), desc=f"Processing {class_name}"):
            image_path = os.path.join(class_path, image_name)
            labels.append(class_to_index[class_name])
            try:
                    img_input = preprocess_image_torch(image_path, h, w)
                    if input_details[0]['dtype'] == np.int8:
                        scale, zp = input_details[0]['quantization']
                        img_input = np.clip(np.round(img_input / scale + zp), -128, 127).astype(np.int8)
                    
                    interpreter.set_tensor(input_details[0]['index'], img_input)
                    interpreter.invoke()
                    output_data = interpreter.get_tensor(output_details[0]['index'])
                    
                    if output_details[0]['dtype'] == np.int8:
                        scale, zero_point = output_details[0]['quantization']
                        output_data = (output_data.astype(np.float32) - zero_point) * scale
                    
                    prediction.append(np.argmax(output_data[0]))
                    
            except Exception as e:
                print(f"Error processing {image_path}: {e}")
    
    print("\n TFLite Inference:")
    print("Top-1 Accuracy:", accuracy_score(labels, prediction))
    print("Classification Report: tflite with quantization\n", classification_report(labels, prediction, target_names=class_names))
     
    cm_tflite = confusion_matrix(labels, prediction)
    sns.heatmap(cm_tflite, annot=True, cmap='Greens', fmt='d', xticklabels=class_names, yticklabels=class_names)
    plt.title("TFLite Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()
    return
    
# evaluation of tflite model

def main():

    try:
        reset_backend()
        inference_tflite()
    except Exception as e:
        print(f"Error processing model {model_path}: {e}")
    
if __name__ == "__main__":
    main()

##    13. Convert TFlite to CC file <a class="anchor" id="head-13"></a>
converting the exported tflite model to a .cc format to deploy in MCU RA8D1 boards. The converted file can be located at `./export folder`.

In [ ]:
# to print the unique operations used in the model file.
def format_tflm_op_name(op_name: str) -> str:
    """
    Formats the TensorFlow Lite Micro operation name by capitalizing the first letter of each word.   
    Args:
        op_name (str): The original operation name.
    Returns:
        str: The formatted operation name.
    """
    parts = op_name.lower().split('_') # Split operation name by underscores
    formatted_parts = [parts[0].capitalize()] + [part.capitalize() for part in parts[1:]] # Capitalize all parts    
    return ''.join(formatted_parts) # Join to match TFLM op style (e.g., Conv2D)

def unique_operation_names(target_tflite_path):
    """
    Extracts and prints all unique operation names used in a given TFLite model.
    Args:
        target_tflite_path (str): Full path to the .tflite model.
    Output:
        Prints a formatted list of unique operation names in the model.
    """
    interpreter = tf.lite.Interpreter(model_path=target_tflite_path) # Load TFLite model
    interpreter.allocate_tensors() # Prepare the interpreter
    op_details = interpreter._get_ops_details() # Internal API to fetch operation details

    # Extract operation names and format them for readability
    unique_op_names = list(set(op['op_name'] for op in op_details))
    formatted_op_names = [format_tflm_op_name(op) for op in unique_op_names]

    # Display the unique operation names
    print("Unique operation names:",formatted_op_names)

# Calling a utility function (assumed pre-defined) that:
# - Parses the .tflite model
# - Extracts and displays a list of all unique operation names (like Conv2D, DepthwiseConv2D, Relu, etc.)
unique_operation_names(os.path.join(os.environ["LOCAL_RESULTS_DIR"],"retrain/export",os.environ["export_tflite_modelname"]))

In [ ]:
# define model name as env variable
%env cc_modelname = model.cc
# Define the shell command as a list of parts
# This command:
# - Runs `xxd -i` to convert the .tflite file into a C array
# - Redirects the output into model.cc
command = ['xxd','-i',os.path.join(os.environ["LOCAL_RESULTS_DIR"],"retrain/export",os.environ["export_tflite_modelname"]),'>', os.environ["cc_modelname"]]

# Execute the shell command in the "export" directory
# Note: " ".join(command) converts the list into a single string command
# shell=True allows redirection ('>') to work
subprocess.run(" ".join(command), shell=True, check=True, cwd=os.path.join(os.environ["LOCAL_EXPERIMENT_DIR"],"export") )
print("succesfully converted tflite model to .cc format....!!")

!echo "Model can be found at: $LOCAL_EXPERIMENT_DIR/export"
!ls $LOCAL_EXPERIMENT_DIR/export

## 14. Next Step <a class="anchor" id="head-8"></a>
#### converted .cc file can be verified on E2Studio and deployed on MCU RA8D1 board.